# Forward Pass Visualization

This notebook visualizes how input propagates through the network during a forward pass, showing:
- Which nodes activate in each iteration
- Input sources for each node
- Whether inputs come from radiation or direct connections
- Activation values and strengths

In [1]:
# Setup: Import dependencies and load configuration
import torch
import torchvision
from torchvision import datasets, transforms
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from pathlib import Path
from typing import Dict, List, Any
import warnings
warnings.filterwarnings('ignore')

# Import project modules
from full_model import initialize_model_and_nodestore
from forward_pass_tracer import ForwardPassTracer

# Load configuration
config_path = "configs/config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  Total nodes: {config['graph']['total_nodes']}")
print(f"  Input nodes: {config['graph']['input_nodes']}")
print(f"  Output nodes: {config['graph']['output_nodes']}")
print(f"  Iterations: {config['model']['iterations']}")
print(f"  Vector dim: {config['model']['vector_dim']}")

Configuration loaded:
  Total nodes: 200
  Input nodes: 14
  Output nodes: 10
  Iterations: 3
  Vector dim: 14


In [2]:
# Initialize model and node store
device = config['system']['device']
if device == 'cuda' and not torch.cuda.is_available():
    device = 'cpu'
    print("CUDA not available, using CPU")

model, node_store = initialize_model_and_nodestore(
    qdrant_url=config['qdrant']['url'],
    collection_name=config['qdrant']['collection_name'],
    total_nodes=config['graph']['total_nodes'],
    input_nodes=config['graph']['input_nodes'],
    output_nodes=config['graph']['output_nodes'],
    cardinality=config['graph']['cardinality'],
    radiation_targets=config['graph']['radiation_targets'],
    vector_dim=config['model']['vector_dim'],
    phase_bins=config['model']['phase_bins'],
    mag_bins=config['model']['mag_bins'],
    iterations=config['model']['iterations'],
    activation_threshold=config['model']['activation_threshold'],
    gamma=config['model']['gamma'],
    device=device,
    temporal_decay=config['model'].get('temporal_decay', 1.0),
    radiation_similarity_threshold=config['model'].get('radiation_similarity_threshold', 0.0),
)

model = model.to(device)
print(f"Model initialized on {device}")

Model initialized on cuda


In [4]:
# Create sample input and run forward pass with tracing
# Generate a random input (or use a specific input pattern)
input_nodes = config['graph']['input_nodes']
vector_dim = config['model']['vector_dim']

T = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((14, 14)),
    transforms.Lambda(lambda x: x.squeeze())
    ])
dataset = datasets.MNIST(root='./data', train=True, transform=T, download=True)
sample_input = dataset[0][0]
sample_output = dataset[0][1]

print(f"Sample input shape: {sample_input.shape}")
print(f"Input range: [{sample_input.min().item():.3f}, {sample_input.max().item():.3f}]")

# Reset model
model.reset()

# Run forward pass with tracer
with ForwardPassTracer() as tracer:
    output = model(sample_input, tracer=tracer)
    trace_data = tracer.get_trace()

print(f"\nForward pass completed!")
print(f"Output shape: {output.shape}")
print(f"Number of iterations traced: {len(trace_data['iterations'])}")

Sample input shape: torch.Size([14, 14])
Input range: [0.000, 0.869]
inactive_output_nodes:  []

Forward pass completed!
Output shape: torch.Size([10])
Number of iterations traced: 3


## Text-Based Visualizations

In [6]:
# Function to display iteration summary
def plot_iteration_summary(trace_data: Dict[str, Any]):
    """Display a summary table of active nodes per iteration."""
    iterations = trace_data['iterations']
    
    summary_data = []
    prev_active_nodes = set()
    
    for i, iter_data in enumerate(iterations):
        current_active_nodes = set(iter_data['active_nodes'])
        
        # New nodes are those in current iteration that weren't in previous
        new_nodes = current_active_nodes - prev_active_nodes
        new_nodes_count = len(new_nodes)
        
        # Nodes that were updated (have details) in this iteration
        nodes_updated = len(iter_data.get('node_details', {}))
        
        summary_data.append({
            'Iteration': iter_data['iteration'],
            'Input Injected': 'Yes' if iter_data['input_injected'] else 'No',
            'Active Nodes': len(current_active_nodes),
            'New Nodes': new_nodes_count,
            'Nodes Updated': nodes_updated,
            'Radiation Connections': sum(len(targets) for targets in iter_data['radiation_targets'].values()),
            'Direct Connections': sum(len(targets) for targets in iter_data.get('direct_connections', {}).values()),
        })
        
        # Update previous active nodes for next iteration
        prev_active_nodes = current_active_nodes
    
    df = pd.DataFrame(summary_data)
    print("Iteration Summary:")
    print("=" * 80)
    print(df.to_string(index=False))
    print("=" * 80)
    print("\nNote: 'New Nodes' = nodes that became active in this iteration")
    print("      'Nodes Updated' = nodes that received inputs and were updated")
    return df

summary_df = plot_iteration_summary(trace_data)

Iteration Summary:
 Iteration Input Injected  Active Nodes  New Nodes  Nodes Updated  Radiation Connections  Direct Connections
         0            Yes            61         61             61                     12                  43
         1             No           143         82            129                     43                 162
         2             No           182         39            171                     89                 357

Note: 'New Nodes' = nodes that became active in this iteration
      'Nodes Updated' = nodes that received inputs and were updated


In [7]:
# Function to display detailed node information for each iteration
def plot_node_details(trace_data: Dict[str, Any], iteration: int = None):
    """Display detailed information about nodes in a specific iteration or all iterations."""
    iterations = trace_data['iterations']
    
    if iteration is not None:
        iterations = [iter_data for iter_data in iterations if iter_data['iteration'] == iteration]
        if not iterations:
            print(f"Iteration {iteration} not found")
            return
    
    for iter_data in iterations:
        print(f"\n{'='*80}")
        print(f"Iteration {iter_data['iteration']} - Detailed Node Information")
        print(f"{'='*80}")
        print(f"Input Injected: {iter_data['input_injected']}")
        print(f"Total Active Nodes: {len(iter_data['active_nodes'])}")
        print(f"\nActive Node IDs: {sorted(iter_data['active_nodes'])}")
        
        node_details = iter_data.get('node_details', {})
        if node_details:
            print(f"\n{'Node ID':<10} {'Input Sources':<30} {'Input Types':<25} {'Activation Strength':<20}")
            print("-" * 85)
            for node_id in sorted(node_details.keys()):
                detail = node_details[node_id]
                input_sources = detail.get('inputs', [])
                input_types = detail.get('input_types', [])
                act_strength = detail.get('activation_strength', 0.0)
                
                # Format input sources and types
                sources_str = str(input_sources) if input_sources else "[] (input node)"
                types_str = str(input_types) if input_types else "[]"
                
                print(f"{node_id:<10} {sources_str:<30} {types_str:<25} {act_strength:<20.4f}")
        else:
            print("No node details available for this iteration")

# Display details for all iterations
plot_node_details(trace_data)


Iteration 0 - Detailed Node Information
Input Injected: True
Total Active Nodes: 61

Active Node IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 20, 25, 34, 36, 39, 45, 46, 52, 54, 56, 59, 60, 61, 63, 67, 75, 76, 80, 87, 88, 98, 99, 100, 102, 103, 120, 126, 131, 132, 143, 145, 151, 152, 159, 160, 163, 166, 168, 170, 174, 175, 176, 180, 182, 186, 198]

Node ID    Input Sources                  Input Types               Activation Strength 
-------------------------------------------------------------------------------------
0          [] (input node)                []                        7.8183              
1          [] (input node)                []                        -1.9659             
2          [] (input node)                []                        1.1026              
3          [] (input node)                []                        2.4738              
4          [] (input node)                []                        -1.0470             
5          [] (in

In [ ]:
# Function to show input source breakdown (radiation vs direct)
def plot_input_breakdown(trace_data: Dict[str, Any]):
    """Show breakdown of radiation vs direct connections per iteration."""
    iterations = trace_data['iterations']
    
    breakdown_data = []
    for iter_data in iterations:
        node_details = iter_data.get('node_details', {})
        
        total_nodes = len(node_details)
        nodes_with_radiation = 0
        nodes_with_direct = 0
        nodes_with_both = 0
        total_radiation_inputs = 0
        total_direct_inputs = 0
        
        for node_id, detail in node_details.items():
            input_types = detail.get('input_types', [])
            radiation_count = input_types.count('radiation')
            direct_count = input_types.count('direct')
            
            if radiation_count > 0 and direct_count > 0:
                nodes_with_both += 1
            elif radiation_count > 0:
                nodes_with_radiation += 1
            elif direct_count > 0:
                nodes_with_direct += 1
            
            total_radiation_inputs += radiation_count
            total_direct_inputs += direct_count
        
        breakdown_data.append({
            'Iteration': iter_data['iteration'],
            'Total Nodes': total_nodes,
            'Nodes w/ Radiation Only': nodes_with_radiation,
            'Nodes w/ Direct Only': nodes_with_direct,
            'Nodes w/ Both': nodes_with_both,
            'Total Radiation Inputs': total_radiation_inputs,
            'Total Direct Inputs': total_direct_inputs,
        })
    
    df = pd.DataFrame(breakdown_data)
    print("Input Source Breakdown (Radiation vs Direct):")
    print("=" * 100)
    print(df.to_string(index=False))
    print("=" * 100)
    return df

breakdown_df = plot_input_breakdown(trace_data)

## Interactive Graph Visualizations

In [ ]:
# Function to create NetworkX graph visualization
def create_activation_graph(trace_data: Dict[str, Any], iteration: int = None, max_nodes: int = 100):
    """
    Create a NetworkX graph showing node activations and connections.
    
    Args:
        trace_data: Trace data from tracer
        iteration: Specific iteration to visualize (None for all)
        max_nodes: Maximum number of nodes to display (for large networks)
    """
    try:
        import plotly.graph_objects as go
        import plotly.express as px
    except ImportError:
        print("Plotly not available, using matplotlib instead")
        use_plotly = False
    else:
        use_plotly = True
    
    iterations = trace_data['iterations']
    if iteration is not None:
        iterations = [iter_data for iter_data in iterations if iter_data['iteration'] == iteration]
    
    # Build graph from all iterations
    G = nx.DiGraph()
    node_colors = {}
    node_sizes = {}
    edge_colors = []
    edge_types = []
    
    # Color scheme
    input_color = '#2ecc71'  # Green for input nodes
    output_color = '#e74c3c'  # Red for output nodes
    regular_color = '#3498db'  # Blue for regular nodes
    radiation_edge_color = '#e67e22'  # Orange for radiation
    direct_edge_color = '#34495e'  # Dark gray for direct
    
    input_nodeids = set(range(config['graph']['input_nodes']))
    output_nodeids = set(range(config['graph']['total_nodes'] - config['graph']['output_nodes'], 
                               config['graph']['total_nodes']))
    
    for iter_data in iterations:
        iter_num = iter_data['iteration']
        node_details = iter_data.get('node_details', {})
        radiation_targets = iter_data.get('radiation_targets', {})
        direct_connections = iter_data.get('direct_connections', {})
        
        # Add nodes
        for node_id in iter_data['active_nodes']:
            if node_id not in G:
                G.add_node(node_id)
                
                # Set node color based on type
                if node_id in input_nodeids:
                    node_colors[node_id] = input_color
                elif node_id in output_nodeids:
                    node_colors[node_id] = output_color
                else:
                    node_colors[node_id] = regular_color
                
                # Set node size based on activation strength
                if node_id in node_details:
                    act_strength = abs(node_details[node_id].get('activation_strength', 0.0))
                    node_sizes[node_id] = max(10, min(50, act_strength * 10))
                else:
                    node_sizes[node_id] = 20
        
        # Add direct connections
        for source, targets in direct_connections.items():
            for target in targets:
                if source in G and target in G:
                    if not G.has_edge(source, target):
                        G.add_edge(source, target)
                        edge_colors.append(direct_edge_color)
                        edge_types.append('direct')
        
        # Add radiation connections
        for source, targets in radiation_targets.items():
            for target in targets:
                if source in G and target in G:
                    if not G.has_edge(source, target):
                        G.add_edge(source, target)
                        edge_colors.append(radiation_edge_color)
                        edge_types.append('radiation')
    
    # Limit nodes if too many
    if len(G.nodes()) > max_nodes:
        # Keep input, output, and most connected nodes
        node_degrees = dict(G.degree())
        important_nodes = (input_nodeids | output_nodeids | 
                          set(sorted(node_degrees.keys(), key=lambda x: node_degrees[x], reverse=True)[:max_nodes]))
        G = G.subgraph(important_nodes).copy()
        node_colors = {n: node_colors.get(n, regular_color) for n in G.nodes()}
        node_sizes = {n: node_sizes.get(n, 20) for n in G.nodes()}
    
    # Create layout
    pos = nx.spring_layout(G, k=1, iterations=50, seed=42)
    
    if use_plotly:
        # Plotly visualization
        edge_trace_direct = []
        edge_trace_radiation = []
        
        for edge in G.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            edge_idx = list(G.edges()).index(edge)
            edge_type = edge_types[edge_idx] if edge_idx < len(edge_types) else 'direct'
            
            edge_trace = go.Scatter(
                x=[x0, x1, None], y=[y0, y1, None],
                line=dict(width=0.5, color=radiation_edge_color if edge_type == 'radiation' else direct_edge_color),
                hoverinfo='none',
                mode='lines',
                showlegend=False
            )
            
            if edge_type == 'radiation':
                edge_trace_radiation.append(edge_trace)
            else:
                edge_trace_direct.append(edge_trace)
        
        node_x = [pos[node][0] for node in G.nodes()]
        node_y = [pos[node][1] for node in G.nodes()]
        node_info = [f"Node {node}<br>Size: {node_sizes.get(node, 20):.1f}" for node in G.nodes()]
        
        node_trace = go.Scatter(
            x=node_x, y=node_y,
            mode='markers+text',
            hoverinfo='text',
            text=[str(n) for n in G.nodes()],
            textposition="middle center",
            hovertext=node_info,
            marker=dict(
                size=[node_sizes.get(node, 20) for node in G.nodes()],
                color=[node_colors.get(node, regular_color) for node in G.nodes()],
                line=dict(width=2, color='white')
            )
        )
        
        fig = go.Figure(data=edge_trace_direct + edge_trace_radiation + [node_trace],
                       layout=go.Layout(
                           title=f'Network Activation Graph (Iteration {iteration if iteration is not None else "All"})',
                           showlegend=False,
                           hovermode='closest',
                           margin=dict(b=20, l=5, r=5, t=40),
                           annotations=[dict(
                               text="Green: Input nodes | Red: Output nodes | Blue: Regular nodes<br>Orange edges: Radiation | Gray edges: Direct",
                               showarrow=False,
                               xref="paper", yref="paper",
                               x=0.005, y=-0.002,
                               xanchor="left", yanchor="bottom",
                               font=dict(size=12)
                           )],
                           xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                           yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
                       ))
        fig.show()
    else:
        # Matplotlib fallback
        plt.figure(figsize=(16, 12))
        
        # Draw edges
        for edge in G.edges():
            edge_idx = list(G.edges()).index(edge)
            edge_type = edge_types[edge_idx] if edge_idx < len(edge_types) else 'direct'
            color = radiation_edge_color if edge_type == 'radiation' else direct_edge_color
            nx.draw_networkx_edges(G, pos, edgelist=[edge], edge_color=color, alpha=0.3, width=0.5, arrows=True)
        
        # Draw nodes
        nx.draw_networkx_nodes(G, pos, 
                              node_color=[node_colors.get(node, regular_color) for node in G.nodes()],
                              node_size=[node_sizes.get(node, 20) * 10 for node in G.nodes()],
                              alpha=0.9)
        
        # Draw labels
        nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')
        
        plt.title(f'Network Activation Graph (Iteration {iteration if iteration is not None else "All"})')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    
    return G, pos

# Create graph for all iterations
graph, positions = create_activation_graph(trace_data)

In [ ]:
# Function to create activation heatmap
def plot_activation_heatmap(trace_data: Dict[str, Any]):
    """Create a heatmap showing activation strength over iterations."""
    iterations = trace_data['iterations']
    
    # Collect all unique node IDs
    all_node_ids = set()
    for iter_data in iterations:
        all_node_ids.update(iter_data['active_nodes'])
    all_node_ids = sorted(list(all_node_ids))
    
    # Build activation matrix
    activation_matrix = []
    iteration_labels = []
    
    for iter_data in iterations:
        iteration_labels.append(f"Iter {iter_data['iteration']}")
        row = []
        node_details = iter_data.get('node_details', {})
        
        for node_id in all_node_ids:
            if node_id in node_details:
                act_strength = abs(node_details[node_id].get('activation_strength', 0.0))
                row.append(act_strength)
            else:
                row.append(0.0)
        
        activation_matrix.append(row)
    
    activation_matrix = np.array(activation_matrix)
    
    # Create heatmap
    plt.figure(figsize=(max(12, len(all_node_ids) * 0.3), max(6, len(iterations) * 0.8)))
    
    # Limit display if too many nodes
    max_display_nodes = 100
    if len(all_node_ids) > max_display_nodes:
        # Show subset of nodes
        node_indices = np.linspace(0, len(all_node_ids) - 1, max_display_nodes, dtype=int)
        activation_matrix = activation_matrix[:, node_indices]
        node_labels = [all_node_ids[i] for i in node_indices]
    else:
        node_labels = all_node_ids
    
    im = plt.imshow(activation_matrix, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(im, label='Activation Strength')
    plt.xlabel('Node ID')
    plt.ylabel('Iteration')
    plt.yticks(range(len(iteration_labels)), iteration_labels)
    plt.xticks(range(len(node_labels)), node_labels, rotation=90, fontsize=8)
    plt.title('Activation Strength Heatmap Across Iterations')
    plt.tight_layout()
    plt.show()
    
    return activation_matrix

activation_matrix = plot_activation_heatmap(trace_data)

In [ ]:
# Function to visualize a specific iteration in detail
def visualize_iteration(trace_data: Dict[str, Any], iteration: int):
    """Create detailed visualization for a specific iteration."""
    iter_data = trace_data['iterations'][iteration] if iteration < len(trace_data['iterations']) else None
    
    if iter_data is None:
        print(f"Iteration {iteration} not found")
        return
    
    print(f"\n{'='*80}")
    print(f"Detailed Visualization for Iteration {iteration}")
    print(f"{'='*80}\n")
    
    # Summary
    print(f"Input Injected: {iter_data['input_injected']}")
    print(f"Active Nodes: {len(iter_data['active_nodes'])}")
    print(f"Radiation Connections: {sum(len(t) for t in iter_data['radiation_targets'].values())}")
    print(f"Direct Connections: {sum(len(t) for t in iter_data.get('direct_connections', {}).values())}")
    
    # Create graph for this iteration
    print(f"\nCreating graph visualization...")
    graph, pos = create_activation_graph(trace_data, iteration=iteration)
    
    # Show node details
    print(f"\nNode Details:")
    plot_node_details(trace_data, iteration=iteration)

# Visualize first iteration (input injection)
visualize_iteration(trace_data, 0)

In [ ]:
# Additional analysis: Track propagation path
def analyze_propagation_path(trace_data: Dict[str, Any], target_node_id: int):
    """Analyze how a specific node gets activated through iterations."""
    print(f"\n{'='*80}")
    print(f"Propagation Path Analysis for Node {target_node_id}")
    print(f"{'='*80}\n")
    
    for iter_data in trace_data['iterations']:
        iter_num = iter_data['iteration']
        node_details = iter_data.get('node_details', {})
        
        if target_node_id in node_details:
            detail = node_details[target_node_id]
            print(f"Iteration {iter_num}:")
            print(f"  Activation Strength: {detail.get('activation_strength', 0.0):.4f}")
            print(f"  Input Sources: {detail.get('inputs', [])}")
            print(f"  Input Types: {detail.get('input_types', [])}")
            
            # Count input types
            input_types = detail.get('input_types', [])
            radiation_count = input_types.count('radiation')
            direct_count = input_types.count('direct')
            print(f"  Radiation inputs: {radiation_count}, Direct inputs: {direct_count}")
            print()

# Analyze a few nodes as examples
if len(trace_data['iterations']) > 0:
    first_iter = trace_data['iterations'][0]
    if first_iter['active_nodes']:
        # Analyze first few active nodes
        for node_id in sorted(first_iter['active_nodes'])[:3]:
            analyze_propagation_path(trace_data, node_id)